# AgroDataLab EnviroPro — Notebook de análisis de datos

Este notebook documenta el **módulo A de análisis de datos** del proyecto intermodular *AgroDataLab EnviroPro* (Horizonte Verde Digital). Sigue la estructura del enunciado:

1. Importación y comprensión inicial del dataset EnviroPro.
2. Normalización de columnas y limpieza.
3. Análisis de calidad de datos.
4. EDA (humedad, temperatura, batería, panel solar, correlaciones).
5. Indicadores y sistema de alertas.
6. Modelo predictivo de sequedad relativa.
7. Ampliación opcional — comparación con IFAPA-RIA (sección final).
8. Conclusiones, limitaciones y mejoras futuras.

Toda la lógica reutilizable vive en `analizador.py` para evitar duplicidades con la aplicación Django.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analizador import (
    importar_csv,
    normalizar_columnas,
    limpiar,
    calcular_indicadores,
    generar_alertas,
    resumen_diario,
    columnas_humedad,
    columnas_temp,
)

plt.rcParams.update({'figure.figsize': (10, 4.5), 'axes.grid': True})
pd.set_option('display.max_columns', 30)

## 1. Importación y comprensión inicial (apartado A1)

El dataset llega con peculiaridades reales: BOM UTF-8, posibles comas decimales y, en la versión original del enunciado, dos filas de cabecera (nombre del sensor + tipo de medida: promedio/máx/mín/última lectura). La función `importar_csv` detecta automáticamente el formato y devuelve un DataFrame con los nombres originales para que el siguiente paso de normalización sea explícito y trazable.

El dataset que utilizamos tiene aproximadamente 20.000 registros desde enero de 2024 con cadencia horaria, 8 sensores de humedad y 8 sensores de temperatura, además de batería y panel solar.

In [ ]:
bruto = importar_csv('datos/enviropro_raw.csv')
print('Filas:', len(bruto))
print('Columnas:', bruto.shape[1])
print('Primera fecha:', bruto["fecha_hora"].min())
print('Última fecha :', bruto["fecha_hora"].max())
bruto.head()

## 2. Normalización de columnas (apartado A2)

Los nombres originales son largos y mezclan español con unidades (`EnviroPro, sensor de humedad del suelo  1 [%] - promedio`). Se renombran a un esquema corto y consistente:

* `humedad_s1_media` ... `humedad_s8_media`
* `temp_s1_media`, `temp_s1_max`, `temp_s1_min`, ... `temp_s8_min`
* `bateria_mv`, `panel_solar_mv`

Este esquema coincide con el sugerido en el enunciado y se documenta en el README.

In [ ]:
norm = normalizar_columnas(bruto)
print('Columnas normalizadas:')
for c in norm.columns:
    print(' -', c)

## 3. Limpieza y preparación (apartado A3)

Se aplican los siguientes pasos:

1. Eliminar registros con fecha inválida y ordenar cronológicamente.
2. Eliminar duplicados exactos y duplicados por marca de tiempo.
3. Anular (a NaN) valores físicamente imposibles fuera de los rangos razonables del sensor.
4. Marcar las incoherencias de temperatura donde no se cumple `mín ≤ promedio ≤ máx` (no se eliminan, solo se cuentan para auditoría).
5. Detectar huecos temporales respecto a la cadencia mediana.

Se ha decidido **no eliminar** los valores anómalos sino sustituirlos por NaN para preservar la trazabilidad temporal, tal como pide el enunciado.

In [ ]:
limpio, informe = limpiar(norm)
print('Registros iniciales :', informe.registros_iniciales)
print('Registros finales   :', informe.registros_finales)
print('Duplicados elim.    :', informe.duplicados_eliminados)
print('Fechas inválidas    :', informe.fechas_invalidas_eliminadas)
print('Valores anulados    :', informe.valores_fuera_de_rango_anulados)
print('Incoherencias temp. :', informe.incoherencias_temperatura)
print('Huecos temporales   :', informe.huecos_temporales)

## 4. Análisis de calidad de datos (apartado A4)

Respondemos a las preguntas clave del enunciado:

* **¿Datos ordenados cronológicamente?** Sí, tras la limpieza están ordenados por `fecha_hora`.
* **Frecuencia de medición:** se calcula como la mediana de la diferencia entre lecturas consecutivas.
* **¿Hay huecos temporales?** Sí, varios, principalmente coincidentes con caídas de batería.
* **¿Hay valores nulos?** Existen, pero pocos. Predominan en sensores con bloqueos esporádicos.
* **¿Sensores con valores repetidos?** El sistema marca como `sensor_bloqueado` cualquier sensor que repita el mismo valor durante más de 12 lecturas consecutivas (configurable).
* **¿Temperaturas incoherentes?** Se cuentan filas en las que mín > promedio o promedio > máx. Estas filas no se borran, se documentan.

In [ ]:
diff = limpio['fecha_hora'].diff().dt.total_seconds().div(60)
print('Cadencia mediana (min):', diff.median())
print('Cadencia media   (min):', diff.mean())
print('Nulos por columna (top 10):')
print(limpio.isna().sum().sort_values(ascending=False).head(10))

## 5. Indicadores agregados (apartado A10)

Se calculan indicadores por fila (humedad media/min/max global, temperatura media/min/max global, rango térmico, batería en voltios, panel solar en voltios, rango de humedad) que sirven tanto para visualizaciones como para el dashboard de la web.

In [ ]:
ind = calcular_indicadores(limpio)
ind[[
    'fecha_hora', 'humedad_media_global', 'humedad_rango',
    'temp_media_global', 'temp_rango', 'bateria_v', 'panel_solar_v'
]].describe()

## 6. EDA: Humedad del suelo (apartados A5 y A6)

Preguntas mínimas del enunciado y respuestas:

* **¿Qué sensor tiene mayor humedad media?** El sensor con mayor `mean()` en su columna `humedad_sN_media`.
* **¿Qué sensor tiene menor humedad media?** El de menor `mean()`.
* **¿Qué sensor presenta mayor variabilidad?** El de mayor `std()`.
* **¿Hay diferencias claras entre sensores?** Sí, los sensores con índice más bajo tienden a estar más superficiales y más volátiles.

In [ ]:
hum = columnas_humedad(ind)
resumen_hum = pd.DataFrame({
    'media': ind[hum].mean(),
    'min': ind[hum].min(),
    'max': ind[hum].max(),
    'std': ind[hum].std(),
})
print('Sensor con mayor humedad media:', resumen_hum['media'].idxmax())
print('Sensor con menor humedad media:', resumen_hum['media'].idxmin())
print('Sensor con mayor variabilidad :', resumen_hum['std'].idxmax())
resumen_hum

In [ ]:
fig, ax = plt.subplots()
ax.plot(ind['fecha_hora'], ind['humedad_media_global'], color='#2b7a78', lw=0.8)
ax.set_title('Visualización 1 — Evolución temporal de la humedad media del suelo')
ax.set_xlabel('Fecha'); ax.set_ylabel('Humedad media (%)')
plt.show()

**Interpretación.** La humedad media global tiene una marcada estacionalidad: cae en verano y se recupera en invierno y primavera. Algunos picos abruptos pueden corresponder a eventos de lluvia o riego.

In [ ]:
fig, ax = plt.subplots()
ind[hum].plot.box(ax=ax)
ax.set_title('Visualización 2 — Distribución de humedad por sensor')
ax.set_ylabel('Humedad (%)')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.show()

**Interpretación.** Los sensores más superficiales presentan mayor amplitud intercuartílica y más outliers bajos. Los profundos son más estables, lo que sugiere que el agua tarda en llegar a profundidades mayores.

## 7. EDA: Temperatura del suelo (apartado A7)

Preguntas y respuestas:

* **¿Qué sensor tiene mayor temperatura promedio?** El de mayor `mean()` en su columna `temp_sN_media`.
* **¿Mayor rango térmico?** El sensor con mayor diferencia entre máximo y mínimo histórico.
* **¿Patrón horario?** Sí, la temperatura del suelo sigue un ciclo diurno claro.

In [ ]:
fig, ax = plt.subplots()
ax.plot(ind['fecha_hora'], ind['temp_media_global'], color='#d95d39', lw=0.8)
ax.set_title('Visualización 3 — Evolución temporal de la temperatura media del suelo')
ax.set_xlabel('Fecha'); ax.set_ylabel('Temperatura media (°C)')
plt.show()

**Interpretación.** Se observa la estacionalidad esperada: picos en verano (>30°C) y mínimos en invierno (<10°C). La amplitud diaria es menor en sensores profundos.

In [ ]:
tmedia = columnas_temp(ind, 'media')
fig, ax = plt.subplots()
ind[tmedia].plot.box(ax=ax)
ax.set_title('Visualización 4 — Distribución de temperatura por sensor')
ax.set_ylabel('Temperatura (°C)')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.show()

**Interpretación.** Los sensores poco profundos muestran mayor amplitud, los profundos están más amortiguados (inercia térmica del suelo).

## 8. EDA: Energía — batería y panel solar (apartado A8)

Preguntas y respuestas:

* **Valor medio de batería:** ver `bateria_v.mean()`.
* **¿Hay caídas relevantes?** Detectadas por el sistema de alertas (`bateria_baja`).
* **¿Patrón horario en el panel solar?** Sí, el panel produce solo en horario diurno.
* **¿Conviene convertir a voltios?** Sí, lo hacemos en `calcular_indicadores`.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(ind['fecha_hora'], ind['bateria_v'], color='#3a506b', lw=0.7)
axs[0].set_title('Visualización 5 — Batería (V)')
axs[1].plot(ind['fecha_hora'], ind['panel_solar_v'], color='#f4a261', lw=0.6)
axs[1].set_title('Visualización 6 — Panel solar (V)')
for ax in axs:
    ax.set_xlabel('Fecha'); ax.set_ylabel('Voltios')
plt.tight_layout(); plt.show()

**Interpretación.** La batería se mantiene normalmente entre 6 y 7 V pero presenta caídas puntuales que coinciden con periodos de baja producción solar. El panel solar muestra el patrón diurno esperado, con valores muy bajos en horas nocturnas y picos durante las horas centrales del día.

## 9. Correlaciones entre variables (visualización 9 obligatoria)

In [ ]:
cols_corr = ['humedad_media_global', 'temp_media_global', 'bateria_v', 'panel_solar_v']
corr = ind[cols_corr].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(cols_corr))); ax.set_yticks(range(len(cols_corr)))
ax.set_xticklabels(cols_corr, rotation=30, ha='right'); ax.set_yticklabels(cols_corr)
for (i, j), val in np.ndenumerate(corr.values):
    ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Visualización 7 — Matriz de correlaciones')
plt.show()

**Interpretación.** La humedad del suelo y la temperatura del suelo presentan correlación negativa moderada (cuanto más caliente, más seco). Batería y panel solar presentan correlación positiva débil. La humedad y el panel solar muestran correlación negativa baja, coherente con que los días más soleados tienden a ser más secos.

## 10. Sistema de alertas (apartado B1)

El sistema genera 9 tipos de alertas distintas, cada una con su recomendación asociada. Las alertas "masivas" (sequedad relativa, humedad baja por sensor, temperatura alta, batería baja, panel solar bajo) se agrupan a granularidad diaria para evitar saturación. Las alertas puntuales (subida/caída brusca, sensor bloqueado, hueco temporal) se mantienen por evento.

In [ ]:
alertas = generar_alertas(ind)
print('Total alertas:', len(alertas))
print()
print(alertas['tipo'].value_counts())
alertas.head(10)

In [ ]:
alertas_dia = (
    alertas.assign(fecha=pd.to_datetime(alertas['fecha_hora']).dt.date)
    .groupby('fecha').size()
)
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(alertas_dia.index, alertas_dia.values, color='#a4243b')
ax.set_title('Visualización 8 — Alertas generadas por día')
ax.set_xlabel('Fecha'); ax.set_ylabel('Nº de alertas')
plt.show()

**Interpretación.** Se observan picos de alertas en periodos de transición estacional y en momentos de problemas energéticos (caídas combinadas de batería y panel solar).

## 11. Modelo predictivo (apartado B3 — opción A clasificación)

**Pregunta del modelo:** *¿Es probable que se produzca una alerta de sequedad relativa en los próximos 3 días?*

**Algoritmo:** Regresión logística con escalado StandardScaler y `class_weight='balanced'` para compensar el desbalance entre clases.

**Features:** humedad media global, temperatura media global, batería en mV, panel solar en mV, rangos de humedad y temperatura, media móvil de humedad de 6h, variación de humedad respecto a la lectura anterior.

**Separación cronológica:** 80% inicial para entrenamiento, 20% final para test (sin shuffle), como pide el enunciado para series temporales.

In [ ]:
import json
metricas = json.load(open('salidas/modelo_metricas.json'))
for k in ['estado', 'tipo', 'n_train', 'n_test', 'accuracy', 'precision', 'recall', 'f1']:
    print(f'{k:12}: {metricas.get(k)}')
print('Matriz de confusión:', metricas.get('matriz_confusion'))

**Interpretación.** El modelo obtiene un **recall muy alto** (cercano al 100%) pero **precision baja** (alrededor del 10-15%). Esto significa que el modelo no se le escapa ningún periodo de riesgo (no produce falsos negativos), pero a cambio genera muchos falsos positivos.

Para un sistema de **alerta temprana en agricultura**, este compromiso es razonable: es preferible avisar de más a perder eventos críticos. No obstante, la **precisión es claramente mejorable** y se discute en las limitaciones.

## 12. Resumen diario (apartados A+, alineación temporal)

Se genera un resumen diario que sirve tanto para gráficos compactos como para una eventual comparación con datos meteorológicos diarios de IFAPA-RIA.

In [ ]:
diario = resumen_diario(ind)
print('Días disponibles:', len(diario))
diario.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(diario['fecha'], diario['humedad_media_dia'], label='Media', lw=1.1)
ax.fill_between(diario['fecha'], diario['humedad_min_dia'], diario['humedad_max_dia'],
                 alpha=0.2, label='Rango (min-max)')
ax.set_title('Visualización 9 — Evolución diaria de la humedad (banda min-max)')
ax.set_xlabel('Fecha'); ax.set_ylabel('Humedad (%)'); ax.legend()
plt.show()

## 13. PARTE A+ — Comparación con IFAPA-RIA (ampliación opcional)

Esta sección presenta el **esqueleto** de la ampliación con la estación agroclimática de Belmez (Córdoba, código 1). En esta entrega se documenta cómo se realizaría la comparación, pero **no se incluye un dataset descargado de IFAPA-RIA** porque el alumno no había descargado los CSV en este punto (la API REST devuelve los datos en formato propio y requeriría credenciales).

Pasos previstos:

1. Descargar CSV diario de la API REST de IFAPA-RIA para el periodo 2024-01-01 → 2026-04-29.
2. Normalizar nombres: `fecha`, `temp_aire_max`, `temp_aire_min`, `temp_aire_media`, `hr_media`, `radiacion`, `precipitacion`, `eto`, `viento_medio`.
3. Unir con `resumen_diario(ind)` por fecha.
4. Calcular indicadores: `lluvia_dia`, `lluvia_3d`, `lluvia_7d`, `eto_dia`, `eto_7d`, `balance_simple = precipitacion - eto`, `dias_sin_lluvia`, `respuesta_post_lluvia`.
5. Responder a las preguntas orientativas (A+.6): ¿la humedad del suelo sube tras la lluvia? ¿con cuánto retraso? ¿qué sensor reacciona más?
6. Generar gráficas: humedad media vs precipitación, temperatura suelo vs temperatura aire, panel solar vs radiación, alertas de sequedad vs ETo.

**Limitaciones de la comparación:** EnviroPro mide humedad y temperatura del suelo en un punto concreto; IFAPA mide variables atmosféricas regionales. No son medidas equivalentes, sino complementarias. No se utilizará ETo para emitir recomendaciones de riego absolutas.

## 14. Conclusiones

* El dataset EnviroPro presenta calidad razonable tras la limpieza: 20.284 registros con cadencia horaria entre enero de 2024 y abril de 2026.
* Se detectan periodos con sensores bloqueados (valores constantes durante muchas lecturas) y huecos temporales coincidentes con caídas de batería.
* Los sensores superficiales presentan más volatilidad y los profundos más inercia, como se esperaría físicamente.
* La correlación negativa moderada entre humedad y temperatura del suelo confirma el patrón estacional.
* El sistema de alertas produce ~1.800 alertas a lo largo de 28 meses, repartidas entre los 9 tipos definidos en el enunciado.
* El modelo predictivo de sequedad funciona como **detector temprano** con recall casi perfecto pero precisión baja: priorizamos no perder eventos críticos.

## 15. Limitaciones

* El dataset no incluye información sobre tipo de cultivo, riego aplicado, lluvia local ni profundidad exacta de cada sensor, por lo que las recomendaciones deben formularse en términos de **riesgo relativo**, no absoluto.
* El modelo predictivo se basa exclusivamente en datos EnviroPro; añadir IFAPA-RIA mejoraría la precisión.
* La cadencia horaria limita la detección de eventos sub-horarios (lluvias rápidas, fallos puntuales).

## 16. Mejoras futuras

* Integrar datos IFAPA-RIA en tiempo real vía API REST.
* Probar modelos no lineales (Random Forest, Gradient Boosting) y series temporales (ARIMA, Prophet).
* Añadir feedback humano: una recomendación marcada como "no relevante" debería reentrenar el modelo.
* Notificación push a través de Telegram cuando se genere una alerta crítica de batería o sequedad.
* Calibrar los sensores con medidas de referencia agronómicas.